# FULL OUTER JOIN and CROSS JOIN

## Overview

### FULL OUTER JOIN
Returns all rows from both tables, with NULLs filled in wherever there is no match on either side.

```
Left only  |  Both matched  |  Right only
  NULL ←→ right cols    |  all cols    |  left cols ←→ NULL
```

**SQLite:** Does not support `FULL OUTER JOIN` natively. Emulate it with `LEFT JOIN UNION ALL RIGHT JOIN ... WHERE left.key IS NULL` (or `LEFT JOIN UNION ALL LEFT JOIN` with tables swapped, keeping only unmatched right rows).

**PostgreSQL:** Full native support — `FULL OUTER JOIN ... ON`.

### CROSS JOIN
Returns the **cartesian product** — every row in the left table paired with every row in the right table. N × M rows total. No ON clause.

**Legitimate uses of CROSS JOIN:**
- Generating all combinations (e.g. every department × every month for a reporting grid)
- Creating a date spine
- Populating test data

**Accidental cartesian product** (missing ON in an implicit join) is one of the most damaging SQL mistakes — it can silently return billions of rows.

---

In [ ]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(':memory:')
conn.executescript("""
CREATE TABLE patients (
    patient_id INTEGER PRIMARY KEY, first_name TEXT, last_name TEXT,
    dob TEXT, province TEXT, active INTEGER DEFAULT 1
);
CREATE TABLE providers (
    provider_id INTEGER PRIMARY KEY, full_name TEXT, specialty TEXT,
    province TEXT, supervisor_id INTEGER
);
CREATE TABLE departments (
    dept_id INTEGER PRIMARY KEY, dept_name TEXT, building TEXT, head_provider_id INTEGER
);
CREATE TABLE encounters (
    enc_id INTEGER PRIMARY KEY, patient_id INTEGER, provider_id INTEGER,
    dept_id INTEGER, enc_date TEXT, diag_code TEXT, cost_cad REAL, admitted INTEGER DEFAULT 0
);
CREATE TABLE diagnoses (
    diag_code TEXT PRIMARY KEY, description TEXT, category TEXT
);
CREATE TABLE referrals (
    referral_id INTEGER PRIMARY KEY, from_provider INTEGER, to_provider INTEGER,
    patient_id INTEGER, referral_date TEXT, reason TEXT
);
INSERT INTO patients VALUES
  (1,'Aroha','Ngata','1985-03-12','NB',1),(2,'Liam','Chen','1972-11-04','NS',1),
  (3,'Fatima','Al-Rashid','1990-07-22','ON',1),(4,'James','MacLeod','1955-01-30','NB',0),
  (5,'Sofia','Petrov','2001-09-15','BC',1),(6,'Noah','Williams','1968-06-08','AB',1),
  (7,'Mei','Zhang','1995-04-17','ON',1),(8,'Ethan','Tremblay','1980-12-01','QC',0),
  (9,'Priya','Nair','1977-08-25','BC',1),(10,'Marcus','Okafor','1993-05-19','ON',1),
  (11,'Dana','Leblanc','2000-02-14','NB',1);
INSERT INTO providers VALUES
  (10,'Dr. Sarah MacNeil','Cardiology','NB',NULL),
  (11,'Dr. James Wong','Oncology','BC',10),
  (12,'Dr. Fatima Al-Rashid','Family Medicine','ON',10),
  (13,'Dr. Ethan Tremblay','Orthopaedics','QC',10),
  (14,'Dr. Priya Nair','Emergency Medicine','BC',11),
  (15,'Dr. Marcus Okafor','Radiology','ON',11),
  (16,'Dr. Unassigned','Neurology','NB',12);
INSERT INTO departments VALUES
  (1,'Cardiology','Building A',10),(2,'Oncology','Building B',11),
  (3,'Family Medicine','Building C',12),(4,'Orthopaedics','Building A',13),
  (5,'Emergency','Building D',14),(6,'Radiology','Building B',15),(7,'Neurology','Building C',16);
INSERT INTO diagnoses VALUES
  ('J18.9','Pneumonia, unspecified','Respiratory'),
  ('I25.1','Atherosclerotic heart disease','Cardiovascular'),
  ('Z00.0','General adult examination','Preventive'),
  ('M17.1','Primary osteoarthritis, knee','Musculoskeletal'),
  ('J06.9','Acute upper respiratory infection','Respiratory'),
  ('R07.9','Chest pain, unspecified','Cardiovascular'),
  ('I10','Essential hypertension','Cardiovascular'),
  ('R55','Syncope and collapse','Neurological'),
  ('I48.0','Paroxysmal atrial fibrillation','Cardiovascular'),
  ('S52.5','Fracture of lower end of radius','Musculoskeletal'),
  ('M54.5','Low back pain','Musculoskeletal');
INSERT INTO encounters VALUES
  (1,1,10,1,'2023-01-15','J18.9',1850.00,1),(2,2,10,1,'2023-02-20','I25.1',3200.00,1),
  (3,3,12,3,'2023-03-05','Z00.0',120.00,0),(4,4,13,4,'2023-03-18','M17.1',5500.00,1),
  (5,5,14,5,'2023-04-02','J06.9',95.00,0),(6,6,10,1,'2023-05-11','R07.9',780.00,0),
  (7,7,10,1,'2023-06-22','I10',2100.00,1),(8,8,12,3,'2023-07-14',NULL,80.00,0),
  (9,1,14,5,'2023-08-30','R55',410.00,0),(10,3,12,3,'2023-09-12','Z00.0',110.00,0),
  (11,9,10,1,'2023-10-01','I48.0',1750.00,1),(12,10,13,4,'2023-11-03','S52.5',2200.00,1),
  (13,2,12,3,'2023-11-18',NULL,90.00,0),(14,5,13,4,'2023-12-05','M54.5',450.00,0);
INSERT INTO referrals VALUES
  (1,12,10,3,'2023-03-10','Chest pain follow-up'),
  (2,10,11,2,'2023-03-01','Suspected malignancy'),
  (3,14,10,9,'2023-09-05','AF workup'),
  (4,12,13,5,'2023-12-01','Back pain unresponsive to treatment'),
  (5,10,15,6,'2023-06-15','Imaging for chest pain');
""")
conn.commit()
print("Schema ready")

# Add a few extra rows to make FULL OUTER JOIN interesting
conn.execute("INSERT INTO patients VALUES (12,'Test','Orphan','1990-01-01','NB',1)")
conn.execute("INSERT INTO encounters VALUES (15,12,10,1,'2023-12-20','J18.9',500.00,0)")
# patient 12 has an encounter but we'll see both unmatched patients and providers
conn.commit()
print("Added patient 12 (has an encounter) to demonstrate full outer patterns")


---
## FULL OUTER JOIN emulation in SQLite

In [ ]:
# SQLite FULL OUTER JOIN workaround: LEFT JOIN UNION ALL anti-join from right side
print("=== All patients AND all providers (FULL OUTER JOIN emulation) ===")
print("    Showing who has encounters vs who doesn't, from either table")
q = """
-- Left side: all patients with their encounter count
SELECT  'patient'           AS entity_type,
        p.patient_id        AS entity_id,
        p.first_name || ' ' || p.last_name AS name,
        COUNT(e.enc_id)     AS encounters
FROM    patients     AS p
LEFT JOIN encounters AS e ON e.patient_id = p.patient_id
GROUP BY p.patient_id

UNION ALL

-- Right side: providers with NO encounters (the unmatched right rows)
SELECT  'provider'          AS entity_type,
        prov.provider_id    AS entity_id,
        prov.full_name      AS name,
        0                   AS encounters
FROM    providers    AS prov
LEFT JOIN encounters AS e ON e.provider_id = prov.provider_id
WHERE   e.enc_id IS NULL

ORDER BY entity_type, encounters DESC
"""
print(pd.read_sql(q, conn).to_string(index=False))

print()
print("=== PostgreSQL native FULL OUTER JOIN (reference) ===")
print("""
SELECT  p.first_name, p.last_name,
        prov.full_name, prov.specialty
FROM    patients  AS p
FULL OUTER JOIN providers AS prov
    ON  p.patient_id = prov.provider_id   -- hypothetical shared key
WHERE   p.patient_id IS NULL OR prov.provider_id IS NULL;
-- Returns rows unmatched on either side
""")


---
## Practical FULL OUTER JOIN: reconciliation between two datasets

In [ ]:
# Healthcare use case: reconcile diagnoses between two systems
# System A: encounters table (diagnoses actually used)
# System B: diagnoses lookup (all coded diagnoses)
# Find: codes in encounters but missing from lookup, AND codes in lookup never used

print("=== Diagnosis code reconciliation ===")
q = """
-- Codes used in encounters
SELECT  e.diag_code          AS enc_code,
        d.diag_code          AS lookup_code,
        d.description,
        CASE
            WHEN d.diag_code IS NULL THEN 'In encounters, NOT in lookup (orphan)'
            WHEN e.diag_code IS NULL THEN 'In lookup, never used in encounters'
            ELSE                          'Matched'
        END AS reconciliation_status
FROM (SELECT DISTINCT diag_code FROM encounters WHERE diag_code IS NOT NULL) AS e
LEFT JOIN diagnoses AS d ON e.diag_code = d.diag_code

UNION ALL

SELECT  NULL              AS enc_code,
        d.diag_code       AS lookup_code,
        d.description,
        'In lookup, never used in encounters' AS reconciliation_status
FROM    diagnoses AS d
LEFT JOIN (SELECT DISTINCT diag_code FROM encounters WHERE diag_code IS NOT NULL) AS e
    ON  d.diag_code = e.diag_code
WHERE   e.diag_code IS NULL

ORDER BY reconciliation_status, lookup_code
"""
print(pd.read_sql(q, conn).to_string(index=False))


---
## CROSS JOIN: cartesian product for combination generation

In [ ]:
# Use case: generate a reporting grid — all departments × all months
# This ensures every cell appears in a report even with zero activity

import datetime

# Month list (Q1-Q2 2023)
months_q = """
WITH months(month) AS (
    VALUES ('2023-01'),('2023-02'),('2023-03'),
           ('2023-04'),('2023-05'),('2023-06')
)
SELECT d.dept_name, m.month,
       COUNT(e.enc_id)            AS encounters,
       COALESCE(SUM(e.cost_cad),0) AS total_cost
FROM   (SELECT dept_name FROM departments) AS d
CROSS JOIN months AS m
LEFT JOIN encounters AS e
    ON  e.dept_id = (SELECT dept_id FROM departments WHERE dept_name = d.dept_name)
    AND STRFTIME('%Y-%m', e.enc_date) = m.month
GROUP BY d.dept_name, m.month
ORDER BY m.month, d.dept_name
"""
result = pd.read_sql(months_q, conn)
print("=== Department × Month grid (CROSS JOIN ensures all cells present) ===")
print(result.to_string(index=False))

print()
print("Without CROSS JOIN, months with zero encounters would simply be absent from results.")


---
## CROSS JOIN: all province × specialty combinations

In [ ]:
# Generate every possible province × specialty pairing
# Then LEFT JOIN actual providers to see which combinations exist
print("=== All province × specialty combinations (CROSS JOIN coverage check) ===")
q = """
WITH provinces(province) AS (
    VALUES ('NB'),('NS'),('ON'),('BC'),('AB'),('QC')
),
specialties(specialty) AS (
    SELECT DISTINCT specialty FROM providers
)
SELECT  pr.province,
        sp.specialty,
        COUNT(prov.provider_id)  AS providers_in_province
FROM    provinces   AS pr
CROSS JOIN specialties AS sp
LEFT JOIN providers AS prov
    ON  prov.province  = pr.province
    AND prov.specialty = sp.specialty
GROUP BY pr.province, sp.specialty
ORDER BY pr.province, sp.specialty
"""
print(pd.read_sql(q, conn).to_string(index=False))
print()
print("Zeros show specialty gaps in each province — useful for capacity planning.")

conn.close()


---
## Common Pitfalls

**1. SQLite does not support FULL OUTER JOIN natively (pre-v3.39 workarounds needed)**
Even though SQLite added RIGHT JOIN in v3.39.0, FULL OUTER JOIN is still not supported. Always use the `LEFT JOIN UNION ALL anti-join` pattern shown in this notebook for full outer semantics in SQLite.

**2. Accidental cartesian product from a missing ON clause**
`FROM encounters, patients` (implicit join syntax) with no WHERE condition produces N×M rows — 14 encounters × 12 patients = 168 rows, silently. At production scale this can produce billions of rows and bring down a server. Always use explicit `JOIN ... ON` syntax and double-check row counts after any join.

**3. CROSS JOIN row counts grow multiplicatively**
5 departments × 12 months = 60 rows. Fine. But 1,000 patients × 365 days × 10 procedures = 3,650,000 rows. CROSS JOINs on large tables require careful row count estimation before execution. Use a CTE with filtered inputs or add WHERE/LIMIT while testing.

**4. FULL OUTER JOIN followed by WHERE on one side converts it to a semi-join**
`FULL OUTER JOIN ... WHERE left.col IS NOT NULL` eliminates all left-unmatched rows and effectively becomes a RIGHT JOIN. After a FULL OUTER JOIN, filter carefully — both sides may have NULLs.

**5. UNION ALL in the FULL OUTER emulation must use WHERE IS NULL on the second branch**
Omitting the `WHERE right.key IS NULL` on the second LEFT JOIN in the emulation causes matched rows to appear twice — once from the first LEFT JOIN and once from the second. The pattern is: first branch gets all left rows, second branch gets only the unmatched right rows.

**6. CROSS JOIN used as an implicit comma join is easy to read as an accident**
`FROM a, b` looks like a CROSS JOIN and is one. Prefer explicit `CROSS JOIN` syntax so reviewers understand the intent. A comment explaining why a cartesian product is intentional also helps.


---
*sql_methods_library - Samantha McGarrigle*